# 📊 01 - Sample Data Generation

**Run this first to generate realistic ticket data** - 

## What this notebook does:
- ✅ Drops and recreates the Unity Catalog schema for a clean start
- ✅ Generates **completely unstructured, realistic ticket data**
- ✅ Creates messy, natural language descriptions perfect for LLM extraction
- ✅ Saves data to Unity Catalog Delta tables
- ✅ Ready for next notebook: `02_action_extraction.ipynb`

**Prerequisites:** None - this is the starting point

## Data Quality: 100% unstructured and realistic!


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets


In [2]:
# Clean start - Drop and recreate schema
print("🗑️ Step 1: Dropping existing schema for clean start...")

try:
    spark.sql(f"DROP SCHEMA IF EXISTS {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']} CASCADE")
    print("✅ Existing schema dropped")
except Exception as e:
    print(f"ℹ️ Schema drop result: {e}")

# Create fresh schema
print("🏗️ Step 2: Creating fresh Unity Catalog schema...")
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}
COMMENT 'Schema for Databricks ticket classification system using Unity Catalog'
""")

print("✅ Fresh schema created")
print(f"🎯 Schema: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")


🗑️ Step 1: Dropping existing schema for clean start...
✅ Existing schema dropped
🏗️ Step 2: Creating fresh Unity Catalog schema...
✅ Fresh schema created
🎯 Schema: quickstart_catalog_vkm_external.classify_tickets


In [3]:
# Generate completely unstructured, realistic ticket data
print("📝 Step 3: Generating realistic, unstructured ticket data...")

# Realistic ticket descriptions - completely unstructured and messy
ticket_descriptions = [
    "hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.",
    
    "urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.",
    
    "i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?",
    
    "our monitoring alerts are going crazy. everything shows red but the app seems to be working fine. not sure if it's a false positive or if something is actually broken. help?",
    
    "the deployment failed again and now production is down. we rolled back but need to figure out what went wrong. this is the third time this month. we need better testing.",
    
    "customers are reporting that their data is missing after the last update. this is a big problem and we need to investigate immediately. could be a data migration issue.",
    
    "the new security patch broke our authentication. users can't log in and we're getting flooded with support tickets. need to fix this before the security team gets involved.",
    
    "our cloud costs are through the roof this month. something is using way more resources than usual. need to find out what's causing the spike and optimize it.",
    
    "the backup system isn't working and we haven't had a successful backup in 3 days. this is a major risk and we need to fix it before something bad happens.",
    
    "the new microservice is causing memory leaks and crashing the whole system. we need to either fix it or disable it until we can figure out what's wrong."
]

# Generate ticket data
tickets_data = []
for i in range(len(ticket_descriptions)):
    ticket_id = f"TICKET_{i+1:03d}"
    
    # Random assignment details
    assigned_to = random.choice(["john.smith@company.com", "sarah.jones@company.com", "mike.wilson@company.com", "lisa.brown@company.com", "unassigned"])
    assignment_group = random.choice(["Platform Team", "Security Team", "Database Team", "Frontend Team", "DevOps Team", "Unassigned"])
    requested_by = random.choice(["customer@example.com", "internal.user@company.com", "support@company.com", "manager@company.com"])
    priority = random.choice(["1 - Critical", "2 - High", "3 - Medium", "4 - Low"])
    state = random.choice(["New", "In Progress", "Assigned", "Pending"])
    
    tickets_data.append({
        "ticket_id": ticket_id,
        "short_description": f"Issue #{i+1} - {random.choice(['Urgent', 'Critical', 'Important', 'Help needed', 'Problem'])}",
        "description": ticket_descriptions[i],
        "assigned_to": assigned_to,
        "assignment_group": assignment_group,
        "requested_by": requested_by,
        "priority": priority,
        "state": state
    })

print(f"✅ Generated {len(tickets_data)} realistic ticket descriptions")
print("📋 Sample descriptions:")
for i, ticket in enumerate(tickets_data[:3]):
    print(f"\n{i+1}. {ticket['ticket_id']}: {ticket['short_description']}")
    print(f"   Description: {ticket['description'][:100]}...")


📝 Step 3: Generating realistic, unstructured ticket data...
✅ Generated 10 realistic ticket descriptions
📋 Sample descriptions:

1. TICKET_001: Issue #1 - Critical
   Description: hey so our website is super slow today and customers are complaining. i think it might be the databa...

2. TICKET_002: Issue #2 - Urgent
   Description: urgent! the login system is broken again. users can't get in and they're calling support nonstop. th...

3. TICKET_003: Issue #3 - Important
   Description: i need help with the new feature we're building. the api is returning weird errors and i don't know ...


In [4]:
# Create DataFrame and save to Unity Catalog
print("💾 Step 4: Saving data to Unity Catalog...")

# Create DataFrame from ticket data
df_tickets = spark.createDataFrame(tickets_data)

# Save to Unity Catalog Delta table
df_tickets.write.format("delta").mode("overwrite").saveAsTable(TABLES["raw_tickets"])

print("✅ Data saved to Unity Catalog Delta table:")
print(f"🎯 Table: {TABLES['raw_tickets']}")

# Display sample data
print("\n📊 Sample data from Unity Catalog:")
display(df_tickets.limit(3))

# Show summary statistics
print(f"\n📈 Summary Statistics:")
print(f"   Total tickets generated: {df_tickets.count()}")
print(f"   Unique assignment groups: {df_tickets.select('assignment_group').distinct().count()}")
print(f"   Priority distribution:")
df_tickets.groupBy("priority").count().orderBy("priority").show()

print("\n🎯 Ready for next notebook: 02_action_extraction.ipynb")


💾 Step 4: Saving data to Unity Catalog...


✅ Data saved to Unity Catalog Delta table:
🎯 Table: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📊 Sample data from Unity Catalog:


,assigned_to,assignment_group,description,priority,requested_by,short_description,state,ticket_id
0,lisa.brown@company.com,Platform Team,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.,4 - Low,customer@example.com,Issue #1 - Critical,In Progress,TICKET_001
1,lisa.brown@company.com,Unassigned,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.,1 - Critical,internal.user@company.com,Issue #2 - Urgent,In Progress,TICKET_002
2,mike.wilson@company.com,Database Team,i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?,3 - Medium,customer@example.com,Issue #3 - Important,Pending,TICKET_003



📈 Summary Statistics:


   Total tickets generated: 10


   Unique assignment groups: 6
   Priority distribution:


+------------+-----+
|    priority|count|
+------------+-----+
|1 - Critical|    1|
|    2 - High|    5|
|  3 - Medium|    2|
|     4 - Low|    2|
+------------+-----+


🎯 Ready for next notebook: 02_action_extraction.ipynb
